In [1]:
pip install torch torchvision wandb thop

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

class CustomCIFAR10(Dataset):
    def __init__(self, root, train=True, transform=None):
        # We use the raw data but wrap it to show custom logic
        self.dataset = datasets.CIFAR10(root=root, train=train, download=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Transformations (Crucial for 32x32 images)
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# To use it:
# train_ds = CustomCIFAR10(root='./data', train=True, transform=transform_train)
# train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

In [3]:
import torch.nn as nn
import torch.nn.functional as F
from thop import profile

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) # 16x16
        x = self.pool(F.relu(self.conv2(x))) # 8x8
        x = self.pool(F.relu(self.conv3(x))) # 4x4
        x = x.view(-1, 128 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# --- FLOPs Counting ---
model = SimpleCNN()
dummy_input = torch.randn(1, 3, 32, 32)
flops, params = profile(model, inputs=(dummy_input, ))
print(f"FLOPs: {flops / 1e6:.2f} Million")
print(f"Parameters: {params / 1e6:.2f} Million")

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pooling.MaxPool2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
FLOPs: 10.85 Million
Parameters: 0.62 Million


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import wandb

# 1. Setup Device (Crucial for speed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Instantiate the Dataset and DataLoader
# (Assuming you ran the CustomCIFAR10 class code from earlier)
train_ds = CustomCIFAR10(root='./data', train=True, transform=transform_train)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

# 3. Initialize Model, Loss, and Optimizer
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Initialize W&B
wandb.init(project="cifar10-lab-worksheet")
wandb.watch(model, log="all", log_freq=10)

# 5. The Training Loop
epochs = 30
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Log metrics to W&B
    acc = 100. * correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f} - Acc: {acc:.2f}%")

    wandb.log({
        "epoch": epoch + 1,
        "loss": running_loss / len(train_loader),
        "accuracy": acc
    })

wandb.finish()

100%|██████████| 170M/170M [00:20<00:00, 8.42MB/s]


Epoch 1/30 - Loss: 1.4955 - Acc: 45.46%
Epoch 2/30 - Loss: 1.0879 - Acc: 61.20%
Epoch 3/30 - Loss: 0.9199 - Acc: 67.52%
Epoch 4/30 - Loss: 0.8248 - Acc: 70.96%
Epoch 5/30 - Loss: 0.7555 - Acc: 73.59%
Epoch 6/30 - Loss: 0.7110 - Acc: 75.18%
Epoch 7/30 - Loss: 0.6688 - Acc: 76.56%
Epoch 8/30 - Loss: 0.6413 - Acc: 77.49%
Epoch 9/30 - Loss: 0.6106 - Acc: 78.44%
Epoch 10/30 - Loss: 0.5902 - Acc: 79.35%
Epoch 11/30 - Loss: 0.5704 - Acc: 80.10%
Epoch 12/30 - Loss: 0.5465 - Acc: 80.65%
Epoch 13/30 - Loss: 0.5392 - Acc: 81.19%
Epoch 14/30 - Loss: 0.5214 - Acc: 81.81%
Epoch 15/30 - Loss: 0.5140 - Acc: 81.91%
Epoch 16/30 - Loss: 0.4960 - Acc: 82.72%
Epoch 17/30 - Loss: 0.4870 - Acc: 82.81%
Epoch 18/30 - Loss: 0.4802 - Acc: 83.21%
Epoch 19/30 - Loss: 0.4691 - Acc: 83.59%
Epoch 20/30 - Loss: 0.4615 - Acc: 83.91%
Epoch 21/30 - Loss: 0.4557 - Acc: 83.98%
Epoch 22/30 - Loss: 0.4443 - Acc: 84.37%
Epoch 23/30 - Loss: 0.4411 - Acc: 84.45%
Epoch 24/30 - Loss: 0.4286 - Acc: 85.03%
Epoch 25/30 - Loss: 0.422

accuracy,▁▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇█████████████
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,86.01
epoch,30
loss,0.39722
